### Importación de Librerías
Cargamos las herramientas necesarias. Destacamos:
* **SVC:** El clasificador de Vectores de Soporte.
* **GridSearchCV:** Para buscar automáticamente los mejores hiperparámetros.
* **StandardScaler:** Fundamental para SVM, ya que este algoritmo se basa en distancias euclídeas y necesita que todas las variables tengan la misma escala.

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

### Carga del Dataset
Descargamos los datos de estrellas directamente desde el repositorio.
Definimos:
* **y (Target):** La columna 'Star category' (lo que queremos predecir).
* **X (Features):** El resto de columnas (propiedades de la estrella).

In [37]:
# Cargamos el Dataset
url = 'https://github.com/YBIFoundation/Dataset/raw/main/Stars.csv'
star = pd.read_csv(url)

# Separar variables predictoras (X) y objetivo (y)
y = star['Star category']
X = star.drop('Star category', axis=1)

### Preprocesamiento Robusto
Preparamos los datos para la SVM:
1.  **Codificación:** Usamos `get_dummies` para transformar texto en números.
2.  **División:** Separamos Train/Test.
3.  **Escalado:** Normalizamos los datos (media=0, varianza=1). Esto es **obligatorio** en SVM; si no lo hiciéramos, las variables con valores numéricos grandes (como 'Temperature' ~30000) dominarían a las pequeñas ('Luminosity' ~0.001), arruinando el modelo.

In [38]:
# Preprocesamiento

# One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

# División Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalado de datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datos procesados y escalados correctamente.")

Datos procesados y escalados correctamente.


### SVM Lineal (Linear Kernel)
Entrenamos una Máquina de Vectores de Soporte buscando un hiperplano lineal que separe las clases.
* **GridSearch:** Probamos distintos valores de `C` (Regularización).
    * `C` bajo: Margen más amplio, permite más errores (evita sobreajuste).
    * `C` alto: Margen estrecho, intenta clasificar todo perfecto (riesgo de sobreajuste).

In [39]:
# Definimos el rango de parámetros a probar
param_grid_linear = {'C': [0.1, 1, 10, 50, 100]}

# Configuramos GridSearch con kernel lineal
grid_linear = GridSearchCV(
    SVC(kernel='linear', random_state=42),
    param_grid_linear,
    refit=True,
    cv=5,
    verbose=1
)

# Entrenamos con datos escalados
grid_linear.fit(X_train_scaled, y_train)

# Resultados
print(f"\nMejor parámetro C (Lineal): {grid_linear.best_params_}")
y_pred_linear = grid_linear.predict(X_test_scaled)
print(f"\nPrecisión (Accuracy) Lineal: {accuracy_score(y_test, y_pred_linear):.4f}")

Fitting 5 folds for each of 5 candidates, totalling 25 fits

Mejor parámetro C (Lineal): {'C': 1}

Precisión (Accuracy) Lineal: 1.0000


### SVM Polinomial (Polynomial Kernel)
Proyectamos los datos a dimensiones superiores usando una función polinómica para separar datos no lineales.
* **Degree:** Probamos polinomios de grado 2, 3 y 4.
* **Coef0:** Controla cuánto influyen los términos de alto grado frente a los de bajo grado.

In [40]:
# Probamos varios grados de polinomio y valores de C
param_grid_poly = {
    'C': [0.1, 1, 10],
    'degree': [2, 3, 4],
    'coef0': [0, 1]  # coef0 es importante para kernels polinomiales y sigmoidales
}

grid_poly = GridSearchCV(
    SVC(kernel='poly', random_state=42),
    param_grid_poly,
    refit=True,
    cv=5,
    verbose=1
)

grid_poly.fit(X_train_scaled, y_train)

print(f"\nMejores parámetros (Polinomial): {grid_poly.best_params_}")
y_pred_poly = grid_poly.predict(X_test_scaled)
print(f"\nPrecisión (Accuracy) Polinomial: {accuracy_score(y_test, y_pred_poly):.4f}")

Fitting 5 folds for each of 18 candidates, totalling 90 fits

Mejores parámetros (Polinomial): {'C': 10, 'coef0': 1, 'degree': 2}

Precisión (Accuracy) Polinomial: 1.0000


### SVM Gaussiano (RBF Kernel)
Utilizamos el kernel de Base Radial (RBF), que es el más potente para fronteras de decisión complejas y curvas.
* **Gamma:** Define el "alcance" de un solo ejemplo de entrenamiento.
    * `Gamma` alto: Las fronteras se ajustan mucho a cada punto (islas).
    * `Gamma` bajo: Las fronteras son más suaves y generalistas.

In [41]:
# Gamma controla la curvatura del límite de decisión
param_grid_rbf = {
    'C': [0.1, 1, 10, 50, 100],
    'gamma': [1, 0.1, 0.01, 0.001, 'scale', 'auto']
}

grid_rbf = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid_rbf,
    refit=True,
    cv=5,
    verbose=1
)

grid_rbf.fit(X_train_scaled, y_train)

print(f"\nMejores parámetros (RBF/Gaussiano): {grid_rbf.best_params_}")
y_pred_rbf = grid_rbf.predict(X_test_scaled)
print(f"\nPrecisión (Accuracy) RBF: {accuracy_score(y_test, y_pred_rbf):.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits

Mejores parámetros (RBF/Gaussiano): {'C': 50, 'gamma': 0.01}

Precisión (Accuracy) RBF: 1.0000


### Comparativa Final
Reunimos las precisiones (Accuracy) de los tres mejores modelos encontrados para determinar qué kernel se adapta mejor a la geometría del dataset de estrellas.

In [42]:
resultados = pd.DataFrame({
    'Modelo': ['Lineal', 'Polinomial', 'Gaussiano (RBF)'],
    'Mejor Accuracy': [
        accuracy_score(y_test, y_pred_linear),
        accuracy_score(y_test, y_pred_poly),
        accuracy_score(y_test, y_pred_rbf)
    ]
})

print(resultados)

            Modelo  Mejor Accuracy
0           Lineal             1.0
1       Polinomial             1.0
2  Gaussiano (RBF)             1.0
